In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import glob as glob
from matplotlib.ticker import LogLocator, FuncFormatter
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

In [2]:
# Read in the files from the first classifier which was produced in early 2025
files = glob.glob('../data/2026_06_17_for_upload/*.csv')
files.sort()

In [3]:
# Read in each file from the first classifier and extract the number of each cell type
cell_type_dict = {}
for i in files:
    z = i.replace('../data/2026_06_17_for_upload/', '').replace('.csv', '')
    a = pd.read_csv(i)
    b = a['2025_10_13_HH'].value_counts()
    b = dict(b)
    b['Total'] = a.shape[0]
    cell_type_dict[z] = b
cell_type_dict

{'P01_Day_0_image_1': {'Neg': 9027,
  'CD8_T': 1821,
  'CD4_T': 446,
  'Endothelial': 329,
  'HSPC': 282,
  'Stromal': 175,
  'Total': 12080},
 'P01_Day_0_image_2': {'Neg': 5638,
  'CD4_T': 954,
  'CD8_T': 603,
  'HSPC': 349,
  'Endothelial': 116,
  'Stromal': 8,
  'Total': 7668},
 'P01_Day_336_image_1': {'Neg': 72684,
  'Stromal': 9216,
  'HSPC': 6457,
  'CD8_T': 4017,
  'CD4_T': 2594,
  'Endothelial': 1738,
  'Total': 96706},
 'P02_Day_0_image_1': {'Neg': 502,
  'CD4_T': 62,
  'CD8_T': 17,
  'HSPC': 8,
  'Endothelial': 4,
  'Stromal': 4,
  'Total': 597},
 'P03_Day_0_image_1': {'Neg': 54599,
  'CD4_T': 10398,
  'CD8_T': 8602,
  'HSPC': 1973,
  'Stromal': 1822,
  'Endothelial': 1231,
  'Total': 78625},
 'P03_Day_168_image_1': {'Neg': 19697,
  'CD4_T': 4751,
  'CD8_T': 1626,
  'HSPC': 1363,
  'Endothelial': 343,
  'Stromal': 342,
  'Total': 28122},
 'P03_Day_336_image_1': {'Neg': 54984,
  'CD4_T': 14897,
  'HSPC': 4307,
  'Stromal': 3898,
  'CD8_T': 3793,
  'Endothelial': 787,
  'Total'

In [4]:
# Turn the dictionary into a dataframe and calculate the percentage of each cell
cell_type_df = pd.DataFrame(cell_type_dict)
cell_type_df = cell_type_df.T
cell_type_df['PID'] = cell_type_df.index.str.split('_').str[0]
cell_type_df['tp'] = cell_type_df.index.str.split('_').str[1:3].str.join('_')
cell_type_df['tp'] = cell_type_df['tp'].str.replace('Day_', '').str.replace('_image', '')
cell_type_df['pid_tp'] = cell_type_df['PID'] + '_' + cell_type_df['tp']
cell_type_df = cell_type_df.groupby(['pid_tp']).mean().reset_index()
cell_type_df.loc[:, 'Neg':] =  cell_type_df.loc[:, 'Neg':].div(cell_type_df['Total'], axis = 0) * 100

cell_type_df = cell_type_df.loc[:, cell_type_df.columns != 'Total']
cell_type_df['pid'] =  cell_type_df['pid_tp'].str.split('_').str[0]
cell_type_df['tp'] =  cell_type_df['pid_tp'].str.split('_').str[1]
cell_type_df

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
# Create dictionaries which detail the clinical outcome
outcomes = pd.read_excel('../data/Outcomes_master_simplified_updatedJan24_patient_updated.xlsx')
outcomes[['Outcome_C6_revised_Vid', 'Outcome_C12_CC486']] = outcomes[['Outcome_C6_revised_Vid', 'Outcome_C12_CC486']].replace({
        'non-responder_2': 'Non-responder',
        'responder_1': 'Responder'})

outcomes_6 = dict(zip(outcomes['PID'], outcomes['Outcome_C6_revised_Vid']))
outcomes_12 = dict(zip(outcomes['PID'], outcomes['Outcome_C12_CC486']))
outcomes_6

In [ ]:
# Delete the following sample as the alignment was crappy. (i.e many T cells didn't have a nucleus)
cell_type_df = cell_type_df .loc[cell_type_df.index != 'P17_Day_168', :]

In [ ]:
# Create a list of patients which occur more than twice
patient_list = cell_type_df['pid'].value_counts()[lambda x: x >= 2].index.tolist()
patient_list

# Filter the dataframe for patients which occur twice
cell_type_df = cell_type_df.loc[cell_type_df['pid'].isin(patient_list), :]


# Reorder the dataframe to allow for easier plotting
order_dict = {
    'r_r': [],
    'nr_r': [],
    'r_prog': [],
    'nr_nr': [],
    'nr_off' : []
}

# 
for patient in cell_type_df['pid']: 
    a = outcomes_6[patient]
    b = outcomes_12[patient]
    if (a == 'Responder') & (b == 'Responder'):
        order_dict['r_r'].append(patient)
    elif (a == 'Responder') & (b == 'Non-responder'):
        order_dict['r_prog'].append(patient)
    elif (a == 'Non-responder') & (b == 'Responder'):
        order_dict['nr_r'].append(patient)
    elif (a == 'Non-responder') & (b == 'Non-responder'):
        order_dict['nr_nr'].append(patient)
    else:
        order_dict['nr_off'].append(patient)

# Remove duplicates from each list
for key in order_dict:
    order_dict[key] = list(set(order_dict[key]))

# Alphabetically order the lists in the dictionary    
for key in order_dict:
    order_dict[key].sort()
    
# Check results
for key, patients in order_dict.items():
    print(f"{key}: {patients}")

In [ ]:
# Reorder the dataframe based on cell type and remove a superfluous timepoint
cell_type_df = cell_type_df[['HSPC','Stromal', 'Endothelial','Neg','CD4_T', 'CD8_T','pid','tp']]
cell_type_df = cell_type_df.loc[cell_type_df.index != 'P08_Day_140', :]

cell_type_df.index = cell_type_df['pid'] + '_' + cell_type_df['tp']

In [ ]:
# Make graphs

# Define timepoint labels and response colors
xlabel_dict = {'0': 'C1', '168': 'C7', '336': 'C12', 'Prog': 'Prog'}
response_colors = {'Responder': '#0c5386', 'Non-responder': '#f7901e', 'C1': '#CCCCCC', 'nan': '#CCCCCC'}

# Specify the colors of each bar to be plotted
cell_color_dict = {'Neg' : '#39737CFF', 'HSPC': '#3CB7CCFF',  'Stromal' : '#32A251FF', 'Endothelial' : '#FFD94AFF',
                   'CD8_T' :  '#FF7F0FFF', 'CD4_T' : '#B85A0DFF'}

# Organize all patients
all_patients = [p for key in ['r_r', 'nr_r', 'r_prog',  'nr_nr', 'nr_off'] for p in order_dict[key]] # specify the order of the patients to be plotted
print(f"Found {len(all_patients)} patients")

# Create subplot layout
fig, axes = plt.subplots(1, len(all_patients), figsize=(6, 2), sharey=True)
plt.rcParams['pdf.fonttype'] = 42
axes = np.atleast_1d(axes)

# Plot each patient
for idx, patient in enumerate(all_patients):
    ax = axes[idx]
    
    # Get patient samples
    patient_samples = {s.split('_')[-1]: s for s in cell_type_df.index # grab the timepoint and create a dictionary which maps to sample names
                       if (s.startswith(f'{patient}_') or s == patient) 
                       and s.split('_')[-1] in xlabel_dict}
    
    # Determine expected timepoints
    expected_timepoints = ['0', '168', '336' if '336' in patient_samples else 'Prog']
    
    # Plot each timepoint for each patient
    for i, tp in enumerate(expected_timepoints):
        if tp in patient_samples:
            sample_data = cell_type_df.loc[patient_samples[tp]]
            
            # Plot stacked bars
            bottom = 0
            for cell_type in cell_type_df.columns[:6]: # use cell_type_df1 in case a patient is missing a celltype
                ax.bar(i, sample_data[cell_type], bottom=bottom, width=0.8,
                       color=cell_color_dict.get(cell_type, 'gray'))
                bottom += sample_data[cell_type]
                
        else:
            # Mark missing timepoints
            ax.text(i, 10, '*', ha='center', va='center', fontsize=10, color='black')
            
        # Add response status bar for non-C1 timepoints
        if tp == '168' and patient in outcomes_6:
            response_color = response_colors[outcomes_6[patient]]
            ax.bar(i, 0.035, bottom=1.02, width=0.8, clip_on=False,
                   color=response_color, transform=ax.get_xaxis_transform())
        elif tp in ['336', 'Prog'] and patient in outcomes_12:
            response_color = response_colors.get(outcomes_12[patient], '#CCCCCC') if pd.notna(outcomes_12[patient]) else '#CCCCCC'
            ax.bar(i, 0.035, bottom=1.02, width=0.8, clip_on=False,
                   color=response_color, transform=ax.get_xaxis_transform())
            
    # Format subplot
    ax.set(xlim=(-0.5, len(expected_timepoints) - 0.5), ylim=(0, 100),
           xticks=range(len(expected_timepoints)),
           xticklabels=[xlabel_dict[tp] for tp in expected_timepoints],
           title=patient, ylabel='% of cells' if idx == 0 else '')
    ax.tick_params(axis='x', rotation=90, length=1, pad=1, labelsize=7)
    ax.tick_params(axis='y', length=1, pad=1, labelsize=7)
    ax.yaxis.label.set_size(7)
    ax.yaxis.labelpad = 1
    ax.set_title(patient, fontsize=8, pad=10)
    ax.spines[['top', 'right']].set_visible(False)

# Add legends
cell_handles = [Patch(facecolor=cell_color_dict.get(ct, 'gray'), label=ct) 
                for ct in cell_type_df.columns[:6]]
response_handles = [Patch(facecolor=response_colors[s], label=s) 
                    for s in ['Responder', 'Non-responder']]
fig.legend(handles=cell_handles[::-1], bbox_to_anchor=(0.95, 0.4), 
           loc='center left', handlelength=1, fontsize=7, title='Cell Type',handletextpad=0.25, title_fontsize=7)
fig.legend(handles=response_handles, bbox_to_anchor=(0.95, 0.8), 
           loc='center left', handlelength=1, fontsize=7, handletextpad=0.25, title='Response Status', title_fontsize=7)

plt.tight_layout(rect=[0, 0, 0.95, 1])
plt.subplots_adjust(wspace=0.2)
#fig.savefig('../results/stacked_bar_chart_by_patient.pdf', bbox_inches='tight')
plt.show()